# Myllia competition
---

**Authors**: [fsb2210](https://www.kaggle.com/fsb2210)

## CRISPRi Perturbation Prediction: Factorized Vector Network

This notebook implements a **Factorized Vector Network** to predict gene expression responses to CRISPRi perturbations. The architecture explicitly models interactions between perturbation, target Gene, and cell Line context.

## Imports and configuration

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split

import scanpy as sc

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# directory with data files
data_dir = "../data"

RANDOM_STATE = 42

# neural network config
LATENT_DIM = 256
ESM2_DIM = 1280
N_TARGETS = 5127
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
EPOCHS = 5
N_CELLS = 5

# synthetic dataset
TARGET_SUM = 10000
PERCENTILE_THRESHOLD = 5
MIN_CELLS = 20
N_RESAMPLES = 2

# pytorch device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data preparation and embedding alignment

We need to have the order of rows in the matrix with esm2 gene embeddings exactly matching the column order of the target cells in the training dataset. Missing embeddings are handled using a mean imputation.

In [3]:
# train & validation sets
train_df = pd.read_csv(f"{data_dir}/training_data_means.csv")
val_df = pd.read_csv(f"{data_dir}/pert_ids_all.csv")

# ground truth
ground_truth = pd.read_csv(f"{data_dir}/training_data_ground_truth_table.csv")

# load pre-fetched gene embeddings using ESM-2
gene_embeddings_df = pd.read_csv(f"{data_dir}/esm2_gene_embeddings.csv")

# load raw data
adata_raw = sc.read_h5ad(f"{data_dir}/raw/training_cells.h5ad")

Define canonical gene order

In [4]:
# target genes
target_genes = train_df.columns[1:].tolist()
assert len(target_genes) == N_TARGETS, "target gene count mismatch!"

# perturbed genes
perturbed_genes = train_df[~train_df["pert_symbol"].str.contains("non-targeting")].iloc[:,0].tolist()
assert len(perturbed_genes) == 80, "perturbed gene count mismatch!"

Load gene embeddings as a dict

In [5]:
esm2_map = dict(zip(gene_embeddings_df["pert_symbol"], gene_embeddings_df.iloc[:,1:].values))
mean_emb = np.mean(list(esm2_map.values()), axis=0)

In [6]:
# construct gene_esm2_matrix
gene_esm2_rows = []
for gene in target_genes:
    if gene in esm2_map:
        gene_esm2_rows.append(esm2_map[gene])
    else:
        gene_esm2_rows.append(mean_emb)  # fallback

gene_esm2 = np.vstack(gene_esm2_rows)
assert gene_esm2.shape == (5127, 1280), "shape mismatch!"

Prepare perturbed genes ESM2 map

In [7]:
pert_esm2_map = {}
for pert in perturbed_genes:
    if pert in esm2_map:
        pert_esm2_map[pert] = esm2_map[pert]
    else:
        pert_esm2_map[pert] = mean_emb 

Prepare weights and baselines

In [8]:
weight_cols = [f"w_{g}" for g in target_genes]
gene_weights = ground_truth[weight_cols].values
baseline_wmae = ground_truth["baseline_wmae"].values

Split data between training and validation sets

In [9]:
def split_train_val(train_df, gene_weights, baseline_wmae, val_size=0.2, random_state=42):
    n_samples = len(train_df)
    indices = np.arange(n_samples)

    # random split
    train_idx, val_idx = train_test_split(indices, test_size=val_size, random_state=random_state)
    train_data = {
        "df": train_df.iloc[train_idx].reset_index(drop=True),
        "weights": gene_weights[train_idx],
        "baseline_wmae": baseline_wmae[train_idx],
    }
    val_data = {
        "df": train_df.iloc[val_idx].reset_index(drop=True),
        "weights": gene_weights[val_idx],
        "baseline_wmae": baseline_wmae[val_idx],
    }

    return train_data, val_data

In [10]:
if "non-targeting" in train_df["pert_symbol"].unique():
    train_df = train_df[~train_df["pert_symbol"].isin(["non-targeting"])]

train_data, val_data = split_train_val(train_df, gene_weights, baseline_wmae, val_size=0.2, random_state=RANDOM_STATE)

## Dataset and DataLoader

Now we handle per-perturbation weights and gene masking for external data

In [11]:
from torch.utils.data import Dataset, DataLoader

class GenesDataset(Dataset):
    def __init__(self, deg_df, esm2_dict, cell_id, gene_weights=None, baseline_wmae=None, gene_mask=None):
        self.deg_df = deg_df
        self.esm2_dict = esm2_dict
        self.cell_id = cell_id

        self.n_pert = len(deg_df)

        self.targets = deg_df.iloc[:, 1:].values.astype("float32")
        self.symbols = deg_df["pert_symbol"].values

        # weights
        if gene_weights is None:
            # External compute proxy weights
            self.gene_weights = np.ones((self.n_pert, N_TARGETS)).astype("float32")
            # self.gene_weights = compute_proxy_weights(deg_df, method=proxy_weight_method)
        else:
            self.gene_weights = gene_weights
        
        # baseline_wmae
        if baseline_wmae is None:
            self.baseline_wmae = np.ones(N_TARGETS)
        else:
            self.baseline_wmae = baseline_wmae
        
        # gene_mask
        if gene_mask is None:
            self.gene_mask = np.ones((self.n_pert, N_TARGETS)).astype("float32")
        else:
            gene_mask = np.array(gene_mask)
            if gene_mask.shape == (N_TARGETS,):
                self.gene_mask = np.tile(gene_mask, (self.n_pert, 1)).astype("float32")
            else:
                self.gene_mask = gene_mask.astype("float32")

    def __len__(self): return self.n_pert

    def __getitem__(self, idx):
        pert_symbol = self.deg_df["pert_symbol"].iloc[idx]
        return {
            "pert_esm2": torch.tensor(self.esm2_dict[pert_symbol], dtype=torch.float32),
            "target": torch.tensor(self.targets[idx], dtype=torch.float32),
            "weights": torch.tensor(self.gene_weights[idx], dtype=torch.float32),
            "baseline_wmae": torch.tensor(self.baseline_wmae[idx], dtype=torch.float32),
            "gene_mask": torch.tensor(self.gene_mask[idx]),
            "cell_id": torch.tensor(self.cell_id),
        }

In [12]:
def collate_fn(batch):
    pert_esm2 = torch.stack([item["pert_esm2"] for item in batch])
    target = torch.stack([item["target"] for item in batch])
    weights = torch.stack([item["weights"] for item in batch])
    gene_mask = torch.stack([item["gene_mask"] for item in batch])
    cell_id = torch.stack([item["cell_id"] for item in batch])
    baseline_wmae = torch.stack([item["baseline_wmae"] for item in batch])
    return {
        "pert_esm2": pert_esm2,
        "target": target,
        "weights": weights,
        "baseline_wmae": baseline_wmae,
        "gene_mask": gene_mask,
        "cell_id": cell_id,
    }

Initialize Dataset

In [14]:
real_dataset = GenesDataset(deg_df=train_data["df"], esm2_dict=pert_esm2_map, cell_id=0, gene_weights=train_data["weights"], baseline_wmae=train_data["baseline_wmae"])
val_dataset = GenesDataset(deg_df=val_data["df"], esm2_dict=pert_esm2_map, cell_id=0, gene_weights=val_data["weights"], baseline_wmae=val_data["baseline_wmae"])
## train_loader = DataLoader(real_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

Generate synthetic perturbations

In [15]:
def generate_loo_synthetics(adata_raw, target_genes, pert_esm2_map,
    exclude_perturbations=None, percentile_threshold=5, n_resamples=5, min_cells=20):
    """
    Generate LOO synthetic perturbations from control cells

    Returns:
        deg_df: DataFrame [N_synth, 5128] (pert_symbol + 5127 targets)
        gene_weights: [N_synth, 5127] (confidence-based)
        gene_mask: [N_synth, 5127] (all 1.0 for Challenge data)
    """
    # isolate controls
    controls = adata_raw[adata_raw.obs["sgrna_symbol"] == "non-targeting"]
    controls = controls[:, target_genes]  # Subset to 5127 targets

    # norm
    X_raw = controls.X.toarray()
    totals = X_raw.sum(axis=1, keepdims=True)
    totals[totals == 0] = 1
    ctrl = np.log2(1.0 + X_raw / totals * TARGET_SUM)

    n_ctrl = controls.shape[0]
    synthetics = []
    pbar = tqdm(enumerate(target_genes), desc="synth. DE", total=len(target_genes), initial=0)
    for i, gene in pbar:
        if exclude_perturbations and gene in exclude_perturbations: continue

        # skip if no ESM2 embedding (cannot train without perturbation embedding)
        if gene not in pert_esm2_map:
            pbar.set_postfix({"gene": f"{gene} (not found in ESM2 embeddings!)"})
            continue

        # gene expression values
        expr = ctrl[:, i]
        expr_raw = X_raw[:, i]

        # filter cases with many zeros
        ## if np.sum(expr > 0) / len(expr) < 0.1: continue

        # split populations
        threshold = np.percentile(expr, percentile_threshold)
        mask_low = expr <= threshold
        mask_ref = expr >= np.percentile(expr, 50)

        # filter cases with few cells
        n_low = np.sum(mask_low)
        n_ref = np.sum(mask_ref)
        if n_low < min_cells or n_ref < min_cells: continue

        for _ in range(n_resamples):
            # resample from low-expression cells
            boot_low_idx = np.random.choice(np.where(mask_low)[0], size=n_low, replace=True)
            pb_low = ctrl[boot_low_idx].mean(axis=0)  # [5127]
            
            # resample from reference cells
            boot_ref_idx = np.random.choice(np.where(mask_ref)[0], size=n_ref, replace=True)
            pb_ref = ctrl[boot_ref_idx].mean(axis=0)  # [5127]
            
            # delta expressions = log-fold change
            de_vector = pb_low - pb_ref  # [5127]
            
            # confidence score
            z_score = (np.mean(expr_raw[mask_ref]) - np.mean(expr_raw[mask_low])) / (np.std(expr_raw[mask_ref]) + 1e-6)
            confidence = np.clip(z_score / 5.0, 0, 1)

            sample_baseline = np.mean(np.abs(de_vector))

            synthetics.append({
                "pert_symbol": gene,
                "response": de_vector,
                "confidence": confidence,
                "baseline_wmae": sample_baseline,
            })

        pbar.set_postfix({"gene": f"{gene} (done!)"})

    if len(synthetics) == 0: return None, None, None

    # convert to DataFrame
    deg_df = pd.DataFrame({
        "pert_symbol": [s["pert_symbol"] for s in synthetics],
        **{gene: [s["response"][i] for s in synthetics] for i, gene in enumerate(target_genes)}
    })

    # weights = confidence (broadcasted to all genes)
    gene_weights = np.array([s["confidence"] for s in synthetics]).reshape(-1, 1)
    gene_weights = np.tile(gene_weights, (1, N_TARGETS)).astype("float32")

    # mask = all 1.0
    gene_mask = np.ones((len(synthetics), N_TARGETS)).astype("float32")

    # baseline
    baseline_wmae = np.array([s["baseline_wmae"] for s in synthetics]).astype("float32")

    return deg_df, gene_weights, gene_mask, baseline_wmae

In [16]:
# generate synthetic dataset
synth_df, synth_weights, synth_mask, synth_wmae = generate_loo_synthetics(
    adata_raw,
    target_genes,
    esm2_map,
    percentile_threshold=PERCENTILE_THRESHOLD,
    n_resamples=N_RESAMPLES,
    min_cells=MIN_CELLS,
)

print(f"- generated {len(synth_df)} LOO synthetic perturbations")

synth. DE: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5127/5127 [03:02<00:00, 28.04it/s, gene=ZYX (done!)]


- generated 10112 LOO synthetic perturbations


Concatenate Datasets

In [17]:
from torch.utils.data import ConcatDataset

# synthetic dataset
if synth_df is not None:
    synth_dataset = GenesDataset(
        deg_df=synth_df,
        esm2_dict=esm2_map,
        cell_id=0,
        gene_weights=synth_weights,
        baseline_wmae=synth_wmae,
        gene_mask=synth_mask
    )

# combined DataLoader
combined_dataset = ConcatDataset([real_dataset, synth_dataset])
train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"- total training samples: {len(combined_dataset)}")
print(f"- total validation samples: {len(val_dataset)}")

- total training samples: 10176
- total validation samples: 16


## Model architecture

We can now create the model.

Note that `gene_esm2` is registered as a buffer to ensure it moves with the model.

In [18]:
class FactorizedVectorNet(nn.Module):
    def __init__(self, genes_esm2, esm2_dim=1280, latent_dim=256, n_targets=5127, n_cells=5):
        super().__init__()
        self.n_targets = n_targets
        self.latent_dim = latent_dim
        
        self.register_buffer("gene_esm2", torch.tensor(genes_esm2, dtype=torch.float32))
        
        # encoders
        self.pert_encoder = nn.Sequential(
            nn.Linear(esm2_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU()
        )
        self.gene_encoder = nn.Sequential(
            nn.Linear(esm2_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU()
        )
        
        # cell line context
        self.cell_embedding = nn.Embedding(num_embeddings=n_cells, embedding_dim=latent_dim)
        
        # interaction MLP
        self.interaction_mlp = nn.Sequential(
            nn.Linear(6 * latent_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU()
        )
        
        # output projection head
        self.output_head = nn.Linear(latent_dim, 1)
        
    def forward(self, pert_esm2, cell_id):
        if pert_esm2.dim() == 1:
            pert_esm2 = pert_esm2.unsqueeze(0)  # [1280] -> [1, 1280]
        if cell_id.dim() == 0:
            cell_id = cell_id.unsqueeze(0)

        B = pert_esm2.shape[0]
        
        # encode inputs
        E_p = self.pert_encoder(pert_esm2)       # [B, d]
        E_g = self.gene_encoder(self.gene_esm2)  # [5127, d]
        E_c = self.cell_embedding(cell_id)       # [B, d]

        # broadcast to compute all gene interactions per perturbation
        E_p_b = E_p.unsqueeze(1).expand(-1, self.n_targets, -1)
        E_g_b = E_g.unsqueeze(0).expand(B, -1, -1)
        E_c_b = E_c.unsqueeze(1).expand(-1, self.n_targets, -1)

        # factorized interactions
        interaction_pg = E_p_b * E_g_b
        interaction_pc = E_p_b * E_c_b
        interaction_gc = E_g_b * E_c_b
        
        # concatenate signals
        H = torch.cat([
            E_p_b,           # Perturbation main effect
            E_g_b,           # Gene main effect
            E_c_b,           # Cell main effect
            interaction_pg,  # Pert × Gene interaction
            interaction_pc,  # Pert × Cell interaction
            interaction_gc   # Gene × Cell interaction
        ], dim=-1)           # [B, 5127, 6*d]
        
        # process
        H = self.interaction_mlp(H)  # [B, 5127, d]
        
        # output
        pred = self.output_head(H).squeeze(-1)  # [B, 5127]
        return pred

Initialize model

In [19]:
model = FactorizedVectorNet(
    genes_esm2=gene_esm2,
    esm2_dim=ESM2_DIM,
    latent_dim=LATENT_DIM,
    n_targets=N_TARGETS,
    n_cells=N_CELLS,
).to(device)

## Metric-aligned loss function

This implements the weighted MAE + weighted cosine with masking of the challenge

In [20]:
def smoothstep(t, eps=1e-6):
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def gate_smoothstep(x, left=0.0, right=0.2):
    if right <= left: raise ValueError("right must be > left")
    t = (x - left) / (right - left)
    return smoothstep(t)

def weighted_cosine_loss(y_pred, y_true, weights, gene_mask=None, left=0.0, right=0.2, eps=1e-6):
    if gene_mask is None: gene_mask = torch.ones_like(y_pred)
    
    y_pred_m = y_pred * gene_mask
    y_true_m = y_true * gene_mask
    w_m = weights * gene_mask
    
    x = torch.maximum(torch.abs(y_pred_m), torch.abs(y_true_m))
    w_gate = gate_smoothstep(x, left, right)
    w2 = w_gate * w_gate * w_m
    
    num = torch.sum(w2 * y_pred_m * y_true_m, dim=1)
    den_a = torch.sqrt(torch.sum(w2 * y_pred_m * y_pred_m, dim=1) + eps)
    den_b = torch.sqrt(torch.sum(w2 * y_true_m * y_true_m, dim=1) + eps)
    
    cos_sim = num / (den_a * den_b).clamp(min=eps)
    return 1.0 - cos_sim

def metric_aligned_loss(y_pred, y_true, gene_weights, baseline_wmae, 
                        gene_mask=None, alpha=0.5, max_log2=10.0, eps=1e-6):
    if gene_mask is None: gene_mask = torch.ones_like(y_pred)

    masked_weights = gene_weights * gene_mask
    masked_true = y_true * gene_mask
    masked_pred = y_pred * gene_mask
    
    # weighted MAE
    abs_err = torch.abs(masked_pred - masked_true)
    sum_err = torch.sum(abs_err * masked_weights, dim=1)
    sum_w = torch.sum(masked_weights, dim=1).clamp(min=eps)
    pred_wmae = sum_err / sum_w  # [B]
    
    pred_wmae = torch.clamp(pred_wmae, min=eps)
    
    # baseline per perturbation
    baseline_wmae = torch.clamp(baseline_wmae, min=eps)
    
    # log-ratio
    log_terms = torch.log2(baseline_wmae / pred_wmae)
    log_terms = torch.clamp(log_terms, max=max_log2)
    mae_loss = -torch.mean(log_terms)
    
    # cosine
    cos_loss = torch.mean(weighted_cosine_loss(masked_pred, masked_true, masked_weights, gene_mask=gene_mask, eps=eps))
    
    # composite
    total_loss = alpha * mae_loss + (1.0 - alpha) * cos_loss
    return total_loss, mae_loss, cos_loss

## Training loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_val_loss = float('inf')
patience_counter = 0
MAX_PATIENCE = 10
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch in train_loader:
        pert_esm2 = batch["pert_esm2"].to(device)
        target = batch["target"].to(device)
        weights = batch["weights"].to(device)
        baseline_wmae = batch["baseline_wmae"].to(device)
        gene_mask = batch["gene_mask"].to(device)
        cell_id = batch["cell_id"].to(device)

        optimizer.zero_grad()
        pred = model(pert_esm2, cell_id)

        loss, mae_comp, cos_comp = metric_aligned_loss(pred, target, weights, baseline_wmae, gene_mask=gene_mask, alpha=0.5)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # validation
    model.eval()
    val_loss = 0
    val_mae = 0
    val_cos = 0
    with torch.no_grad():
        for batch in val_loader:
            pert_esm2 = batch["pert_esm2"].to(device)
            target = batch["target"].to(device)
            weights = batch["weights"].to(device)
            baseline_wmae = batch["baseline_wmae"].to(device)
            gene_mask = batch["gene_mask"].to(device)
            cell_id = batch["cell_id"].to(device)
            
            pred = model(pert_esm2, cell_id)
            
            loss, mae_comp, cos_comp = metric_aligned_loss(pred, target, weights, baseline_wmae, gene_mask=gene_mask, alpha=0.5)
            
            val_loss += loss.item()
            val_mae += mae_comp.item()
            val_cos += cos_comp.item()
    
    val_loss /= len(val_loader)
    val_mae /= len(val_loader)
    val_cos /= len(val_loader)
    
    scheduler.step(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        patience_counter = 0
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val MAE: {val_mae:.4f} | "
          f"Val Cos: {1-val_cos:.4f} | "
          f"Patience: {patience_counter}/{MAX_PATIENCE}")
    
    if patience_counter >= MAX_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load("best_model.pth"))
print(f"Best validation loss: {best_val_loss:.4f}")

Epoch 1/5 | Train Loss: 0.4036 | Val Loss: 2.0520 | Val MAE: 3.3537 | Val Cos: 0.2497 | Patience: 0/10
Epoch 2/5 | Train Loss: 0.3128 | Val Loss: 2.4601 | Val MAE: 3.3582 | Val Cos: -0.5620 | Patience: 1/10


---

## Inference

In [ ]:
def predict(model, loader, device=device):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            pert_esm2 = batch['pert_esm2'].to(device)
            cell_id = batch['cell_id'].to(device)
            
            pred = model(pert_esm2, cell_id)
            all_preds.append(pred.cpu().numpy())
    return np.vstack(all_preds)

# generate predictions
# predictions = predict(model, train_loader)
# print(f"Prediction Shape: {predictions.shape}")

# save Submission
# submission_df = train_df[['pert_symbol']].copy()
# submission_df.iloc[:, 1:] = predictions  # ensure columns match train_df order
# submission_df.to_csv('submission.csv', index=False)